In [ ]:
library(rtracklayer)
library(infercnv)
library(Seurat)

In [ ]:
### Create gene position file
# Import GTF
gtf <- rtracklayer::import("/data/ebaird/scRNAseq/SCENTINELsep24/refs/genes.gtf")

# Extract genes (filter out non-standard chromosomes)
genes <- as.data.frame(gtf[gtf$type == "gene"])
# Drosophila-specific filter (keeps chr2L/2R/3L/3R/X/4)
genes <- genes[grepl("chr[23][LR]|chrX|chr4", genes$seqnames), ]

# Format for InferCNV (use gene_name if available, otherwise gene_id)
genes$seqnames <- paste0("chr", genes$seqnames)  # Add 'chr' prefix
genes$gene_name <- ifelse(is.na(genes$gene_name), genes$gene_id, genes$gene_name)

# Save
write.table(genes[, c("gene_name", "seqnames", "start", "end")],
            "dmel_gene_positions.txt",
            sep = "\t", col.names = FALSE, row.names = FALSE, quote = FALSE)

In [ ]:
### Load seurat object
seu <- readRDS("/data/ebaird/scRNAseq/SCENTINELsep24/composition_DEG_signatures/signatures.rds")
DefaultAssay(seu) <- "RNA"
Idents(seu) <- "seurat_clusters"

In [ ]:
seu

In [ ]:
library(infercnv)
library(Seurat)

# Run InferCNV
infercnv_obj <- CreateInfercnvObject(
  raw_counts_matrix = GetAssayData(seu, slot = "counts"),
  annotations_file = as.matrix(seu@active.ident),
  gene_order_file = "/data/ebaird/scRNAseq/SCENTINELsep24/refs/dmel_gene_positions.txt",
  delim="\t",
  ref_group_names = NULL  # Omit if no normal cells
)

In [ ]:
infercnv_obj <- infercnv::run(
  infercnv_obj,
  cutoff = 0.1,              # Lower = more sensitive
  out_dir = "/data/ebaird/scRNAseq/SCENTINELsep24/infercnv4", 
  cluster_by_groups = TRUE,  # Cluster by cell groups
  denoise = TRUE,            # Remove noise
  HMM = TRUE,                # Use HMM prediction
  analysis_mode = "subclusters",
  num_threads = 16,
  useRaster = FALSE,
  BayesMaxPNormal = 0.2
)

In [ ]:
infercnv_obj <- readRDS("/data/ebaird/scRNAseq/SCENTINELsep24/infercnv4/run.final.infercnv_obj")

In [ ]:
plot_cnv(infercnv_object, 
          output_filename = "cnv_plot",
          output_format = "png",
          color_safe = TRUE,
          useRaster = FALSE)

In [ ]:
seu <- add_to_seurat(seurat_obj = seu,
                     infercnv_output_path = '/data/ebaird/scRNAseq/SCENTINELsep24/infercnv4')

In [ ]:
FeaturePlot(seu, reduction='umap', features='proportion_scaled_cnv_chr2L') + ggplot2::scale_colour_gradient(low = 'lightgrey', high = 'blue', limits=c(0,1))

In [ ]:
DimPlot(seu, reduction='umap', group.by = 'has_dupli_chr3R', pt.size=0.5)

In [ ]:
seu@assays